In [ ]:
import pandas as pd
import os
from tqdm import tqdm
import pickle

data.txt is a CSV containing the following heading:

Reaction ID,Reaction_Smiles,src,tgt,reaction_mapping_rxn,set


#### Reaction ID is the Reaxys ID

#### Reaction_Smiles is the SMILES of the reaction in the format reactants>>products

#### src is the reactant SMILES

#### tgt is the product SMILES

#### reaction_mapping_rxn is the SMILES of the reaction with atom mapping included

#### set is whether that datapoint is in the train/validation/test set



In [ ]:
result_folder = 'myNERFresults/'
tt = pd.read_csv("data.txt")
test = tt[tt['set'] == 'test']

In [ ]:
pred_all = None
with open("{}".format(result_folder) + '0.7_2019.pickle', 'rb') as file:
    a = pickle.load(file)
target = a['tgt']

for i in tqdm(range(20)):
    filename = '{}/{}_2019.pickle'.format(result_folder, 0.7*1.3**i)
    with open(filename, 'rb') as file:
        tem = pickle.load(file)
    
    if pred_all is None:
        pred_all = tem['pred']
    else:
        for i in range(len(pred_all)):
            if pred_all[i] is None:
                pred_all[i] = tem['pred'][i]
            else:
                if  tem['pred'][i] is not None:
                    pred_all[i] += tem['pred'][i]
                    
def unorder_set(x):
    try:
        return sorted(set(x), key=x.index)
    except:
        return set()

for i in range(len(pred_all)):
    pred_all[i] = unorder_set(pred_all[i])
    
final_pred = []
for i in range(10):
    tem = []
    for a_pred in pred_all:
        try:
            tem.append(a_pred[i])
        except:
            tem.append("")
    final_pred.append(tem)

test_df = pd.DataFrame(target)
test_df.columns = ['target']

for i, preds in enumerate(final_pred):
    test_df['prediction_{}'.format(i + 1)] = preds

def get_rank(row, base, max_rank):
    for i in range(1, max_rank+1):
        if row['target'] == row['{}{}'.format(base, i)]:
            return i
    return 0

test_df['rank'] = test_df.apply(lambda row: get_rank(row, 'prediction_', 10), axis=1)
test_df['src'] = test['src'].values

prediction_1 is the top-1 prediction 

prediction_2 is the top-2 prediction

src: reactant

tgt: ground-truth product

In [ ]:
test_df